In [ ]:
import gc
import os
import subprocess
import sys
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

# Ensure accelerate is up-to-date (required for device_map='balanced')
print("Installing/upgrading accelerate and diffusers...", flush=True)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "accelerate", "diffusers"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"pip install failed: {result.stderr[:200]}", flush=True)
else:
    print("accelerate and diffusers ready", flush=True)


In [ ]:
from diffusers import FluxPipeline
from datetime import datetime
import torch

In [ ]:
import json
import subprocess
import tempfile

HF_TOKEN_PATH = "/kaggle/input/imggenhub-hf-token/hf_token.json"
HF_TOKEN = None
if os.path.exists(HF_TOKEN_PATH):
    with open(HF_TOKEN_PATH, "r") as f:
        HF_TOKEN = json.load(f)["HF_TOKEN"]
    print("HF_TOKEN loaded from mounted dataset")
else:
    print("Dataset not mounted, downloading via kaggle API...")
    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            result = subprocess.run(["kaggle", "datasets", "download", "leventecsibi/imggenhub-hf-token", "-p", tmpdir, "--unzip"], capture_output=True, text=True, timeout=60)
            if result.returncode == 0:
                token_file = os.path.join(tmpdir, "hf_token.json")
                with open(token_file, "r") as f:
                    HF_TOKEN = json.load(f)["HF_TOKEN"]
                print("HF_TOKEN downloaded and loaded successfully")
            else:
                print(f"Failed to download dataset: {result.stderr}")
    except Exception as e:
        print(f"Error downloading token: {e}")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set in environment")
else:
    print("CRITICAL ERROR: Could not load HF_TOKEN")
    raise RuntimeError("HF_TOKEN is required but could not be loaded")

In [ ]:
import re
MODEL_ID = "black-forest-labs/FLUX.1-schnell"
PROMPTS = ["test"]
OUTPUT_DIR = "."
IMG_SIZE = (1024, 1024)
GUIDANCE = 1.5
STEPS = 4
SEED = 42
PRECISION = "bf16"
INDEX_OFFSET = 0
KERNEL_ID = "unknown"
LORA_REPO_ID = None
LORA_FILENAME = None
LORA_SCALE = 0.8

def slugify(text):
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "_", text)
    text = re.sub(r"^-+|-+$", "", text)
    return text[:30]

In [ ]:
import gc

os.makedirs(OUTPUT_DIR, exist_ok=True)

dtype_map = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}
torch_dtype = dtype_map.get(PRECISION, torch.bfloat16)

gpu_count = torch.cuda.device_count()
print(f"Available GPUs: {gpu_count}", flush=True)
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB VRAM", flush=True)

print(f"Loading model with device_map='balanced' across {gpu_count} GPU(s)...", flush=True)
try:
    pipe = FluxPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        device_map="balanced",
        token=HF_TOKEN
    )
    pipe.set_progress_bar_config(disable=False)
    print("Model loaded with device_map='balanced'.", flush=True)
except Exception as e:
    print(f"device_map='balanced' failed: {e}", flush=True)
    print("Falling back to sequential CPU offload...", flush=True)
    pipe = FluxPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        token=HF_TOKEN
    )
    pipe.enable_sequential_cpu_offload()
    pipe.set_progress_bar_config(disable=False)
    print("Fallback model loaded with CPU offload.", flush=True)

if LORA_REPO_ID:
    print(f"Loading LoRA weights from {LORA_REPO_ID} / {LORA_FILENAME} (scale={LORA_SCALE})...", flush=True)
    load_kwargs = {"adapter_name": "photorealism"}
    if LORA_FILENAME:
        load_kwargs["weight_name"] = LORA_FILENAME
    pipe.load_lora_weights(LORA_REPO_ID, token=HF_TOKEN, **load_kwargs)
    pipe.set_adapters(["photorealism"], adapter_weights=[LORA_SCALE])
    print("LoRA weights loaded and activated.", flush=True)
else:
    print("No LoRA specified. Generating without LoRA.", flush=True)

print(f"Model ready. dtype={torch_dtype}, GPUs={gpu_count}", flush=True)


In [ ]:
print(f"Generating {len(PROMPTS)} image(s) at {IMG_SIZE[1]}x{IMG_SIZE[0]}", flush=True)
print(f"guidance={GUIDANCE}, steps={STEPS}, precision={PRECISION}", flush=True)
if LORA_REPO_ID:
    print(f"LoRA: {LORA_REPO_ID} (scale={LORA_SCALE})", flush=True)

# VAE memory optimizations (helps prevent OOM during decode on T4x2)
try:
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    print("VAE slicing/tiling enabled.", flush=True)
except Exception:
    pass

for i, prompt in enumerate(PROMPTS):
    print(f"[{i+1}/{len(PROMPTS)}] {prompt}", flush=True)
    # Use CPU generator to avoid device conflicts with device_map="balanced"
    generator = torch.Generator(device="cpu").manual_seed(SEED + i)
    gc.collect()
    torch.cuda.empty_cache()
    image = pipe(
        prompt,
        height=IMG_SIZE[0],
        width=IMG_SIZE[1],
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        generator=generator,
        max_sequence_length=256,
    ).images[0]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_kernel_name = str(KERNEL_ID).split("/")[-1]
    safe_prompt = slugify(prompt)
    filename = f"gen_{safe_kernel_name}_p{i+1+INDEX_OFFSET}_{safe_prompt}_{timestamp}.png"

    image.save(os.path.join(OUTPUT_DIR, filename))
    print(f"Saved: {filename}", flush=True)
    gc.collect()

print(f"Complete! {len(PROMPTS)} images in {OUTPUT_DIR}", flush=True)
